# MP04 — Image Comparison Tool
Implementation experiments executed; learner mastery remains unassessed. Objective: signed differences, absolute error, MSE and visual interpretation.

## What does error mean?
Let p[h,w,c] be a prediction and r[h,w,c] a reference on the same H×W×C grid, in the same units. Define d=p−r. Positive d means the prediction is larger. Cast integer inputs before subtraction: uint8 cannot represent −255.

For K valid pixels in a band, bias=sum(d)/K; mean absolute error (MAE)=sum(|d|)/K; mean squared error (MSE)=sum(d²)/K. Bias and MAE retain input units; MSE has squared units. Positive and negative errors can cancel in bias, but cannot cancel in MAE or MSE. For errors [−2,+2], bias=0, MAE=2, MSE=4. These statistics answer different questions.

A boolean H×W mask selects valid pixels for every band. An empty mask is an error. Equal array shapes alone do not establish alignment: images must refer to corresponding ground locations, compatible bands, times and radiometric units.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=Path.cwd()
if not (root/'src').exists(): root=root.parent
sys.path.insert(0,str(root/'src'))
from rssr.image_foundations import resize_hwc,compare_hwc
reference=np.array([[[2.],[2.]]])
prediction=np.array([[[0.],[4.]]])
result=compare_hwc(prediction,reference)
for key in ('bias','mae','mse'): print(key,result[key])
np.testing.assert_allclose(result['mse'],4)
print('safe uint8 difference:',compare_hwc(np.zeros((1,1,1),np.uint8),np.full((1,1,1),255,np.uint8))['signed'])

## Controlled comparisons before interpretation
We compare identical images, a brightness increase of +0.02, and a one-pixel horizontal shift. With no clipping, a uniform +0.02 change predicts bias=MAE=0.02 and MSE=0.0004. The shift preserves content but changes correspondence, demonstrating why registration matters.

The real training crop uses the same checksummed source and fixed scaling as MP03. A rolled shift wraps its first column; we exclude that column from **all three** comparisons, giving equal support of 4032 pixels. That exclusion prevents a synthetic wrap boundary from contaminating the comparison. It is not a registration method. No test scene is used.

In [ ]:
x=np.arange(24,dtype=float).reshape(2,4,3)/100
mask=np.ones((2,4),bool); mask[:,0]=False
for label,p in [('identical',x),('brightness',x+.02),('shift',np.roll(x,1,axis=1))]:
    e=compare_hwc(p,x,mask)
    print(label,e['bias'],e['mae'],e['mse'])
np.testing.assert_allclose(compare_hwc(x+.02,x,mask)['mse'],.0004)

## Maps and measured results
Run `python scripts/image_foundations_experiments.py` to generate both projects' evidence. Signed maps use a diverging color scale centered at zero, fixed to [−0.04,+0.04]. Absolute maps use [0,0.04]. All pairs share the same limits; invalid pixels are blank. Averaging signed differences over bands can conceal opposing band errors, so per-band numerical summaries accompany the maps.

Observed: identical images give zero error, brightness gives the predicted 0.0004 MSE per band, and shifted images have nonzero error with near-zero bias. Near-zero bias therefore does not imply a correct image. Changing display limits independently could conceal these differences.

In [ ]:
from IPython.display import display,Image
p=root/'reports/mp03-mp04-experiments.json'
if p.exists():
    print(json.loads(p.read_text())['mp04'])
    display(Image(filename=str(root/'reports/mp04-differences.png')))
else: print('Run the explicit real-data experiment first.')

## Practice
1. For errors [−1,+3], calculate bias, MAE and MSE.
2. Mask out the second pixel. How do all three statistics change?
3. Explain why subtracting images in different physical units is invalid even when their shapes match.
4. Describe how clipping could invalidate the brightness experiment's analytic prediction.

## Solution guide — generated answers, not learner work
1. Bias=1, MAE=2, MSE=5.
2. Only −1 remains: bias=−1, MAE=1, MSE=1.
3. Numerical subtraction requires compatible meanings and units; dimensions alone cannot validate this.
4. Saturated values change by less than the requested offset. The assumption of a constant +0.02 difference no longer holds.

In [ ]:
r=np.zeros((1,2,1)); p=np.array([[[-1.],[3.]]])
e=compare_hwc(p,r)
np.testing.assert_allclose(e['bias'],1); np.testing.assert_allclose(e['mae'],2); np.testing.assert_allclose(e['mse'],5)
e=compare_hwc(p,r,np.array([[True,False]]))
np.testing.assert_allclose(e['mse'],1)
print('Worked solutions checked; learner mastery unassessed.')

## Execution, limitations and references
Use [local/Colab setup](../docs/setup.md), then run this notebook from notebooks/ or the repository root. Module: `src/rssr/image_foundations.py`; tests: `python -m pytest tests/test_image_foundations.py`. No data download happens in this notebook.

This experiment uses a known synthetic shift on one training crop. It does not estimate sensor registration, perceptual quality or geographic generalization. See the [authoritative brief](../stage_01_python_numpy/MP04_image_comparison_tool/README.md), [evaluation protocol](../docs/protocol.md), [dataset card](../docs/dataset-cards.md), and [measured record](../reports/mp03-mp04-experiments.json).